# SIC 2026 · Inteligencia Artificial
## Clase 14: De la neurona a TensorFlow (Capítulo 8, Unidades 1 y 2)

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

La teoría de redes neuronales ya la vieron en el curso introductorio. Hoy la **hacemos funcionar con números y código**, y terminamos entrenando un modelo con TensorFlow.

| Hora | Parte | Láminas | Qué vamos a hacer |
|---|---|---|---|
| 18:00 a 18:30 | 1. El perceptrón | 9 a 26 | OR y AND con los pesos de las láminas, y por qué XOR no se puede |
| 18:30 a 19:00 | 2. El perceptrón multicapa | 28 a 43 | Resolver XOR con una capa oculta |
| 19:00 a 19:30 | 3. Cómo aprende una red | 45 a 59 | Descenso de gradiente a mano, paso a paso, y funciones de activación |
| 19:30 a 19:40 | Pausa | | |
| 19:40 a 20:10 | 4. Problemas de las redes profundas | 61 a 70 | Gradiente que desaparece y dropout |
| 20:10 a 21:10 | 5. TensorFlow esencial | 72 a 109 | Tensores, operaciones y `tf.data` |
| 21:10 a 22:00 | 6. Primer modelo con TensorFlow | 130 a 147 | El descenso de gradiente de la Parte 3, ahora automático |

Cada paso tiene **qué vamos a hacer**, **el código** y **qué deberías ver**. Los ejercicios tienen verificación y las soluciones están al final.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import warnings
warnings.filterwarnings("ignore")

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

print("TensorFlow", tf.__version__)

---
# Parte 1. El perceptrón (láminas 9 a 26)

Un perceptrón hace tres cosas:

1. Multiplica cada entrada por un **peso**: $x_1 w_1$, $x_2 w_2$.
2. Suma todo, más un **sesgo** $w_0$ (en las láminas, una entrada $x_0$ que siempre vale 1): $s = w_0 + x_1 w_1 + x_2 w_2$.
3. Pasa la suma por una **función umbral**: si $s \\geq 0$, la salida es 1; si no, es −1.

Es exactamente la tabla de Excel de las láminas 11 a 17, pero en NumPy.

## Paso 1.1. La operación OR

Las láminas usan $w_0 = -0{,}5$, $w_1 = 1$, $w_2 = 1$.

**Qué vamos a hacer:** calcular la suma y la salida para las 4 combinaciones de la tabla de verdad.

In [ ]:
X = np.array([[0, 0], [1, 0], [0, 1], [1, 1]])      # las 4 filas de la tabla de verdad

def umbral(s):
    return np.where(s >= 0, 1, -1)                   # 1 si s >= 0, si no -1

def perceptron(X, w0, w1, w2):
    s = w0 + X[:, 0] * w1 + X[:, 1] * w2             # sesgo + x1*w1 + x2*w2
    return s, umbral(s)

s, salida = perceptron(X, w0=-0.5, w1=1, w2=1)
pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "suma s": s, "salida": salida, "OR esperado": [-1, 1, 1, 1]})

**Qué deberías ver:** sumas −0,5; 0,5; 0,5 y 1,5, y salidas −1, 1, 1, 1: coincide con OR (−1 es falso y 1 es verdadero). Son los mismos números de la lámina 17.

**Qué significa geométricamente (láminas 18 y 19):** los pesos definen una **recta**, $-0{,}5 + x_1 + x_2 = 0$. Los puntos de un lado dan 1 y los del otro, −1.

In [ ]:
def dibujar_recta(w0, w1, w2, esperado, titulo, ax):
    colores = ["#D9534F" if e == -1 else "#1428A0" for e in esperado]
    ax.scatter(X[:, 0], X[:, 1], c=colores, s=200, zorder=3)
    xs = np.linspace(-0.5, 1.5, 10)
    if w2 != 0:
        ax.plot(xs, -(w0 + w1 * xs) / w2, "--", color="gray")
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.set_title(titulo); ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.grid(alpha=0.3)

fig, ax = plt.subplots(figsize=(4, 4))
dibujar_recta(-0.5, 1, 1, [-1, 1, 1, 1], "OR: una recta separa los colores", ax)
plt.show()

## Paso 1.2. La operación AND

AND solo es verdadero cuando ambas entradas son 1: la salida esperada es −1, −1, −1, 1.

La lámina 24 propone $w_0 = 0{,}5$, $w_1 = -0{,}5$, $w_2 = -0{,}1$. Comprobémoslo.

In [ ]:
s, salida = perceptron(X, w0=0.5, w1=-0.5, w2=-0.1)
pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "suma s": s.round(2), "salida": salida, "AND esperado": [-1, -1, -1, 1]})

**Qué deberías ver:** salidas 1, 1, 1, −1. **¡Es justo lo contrario de AND!** Esos pesos calculan **NAND** (no-AND). La lámina tiene un error: es un buen ejemplo de por qué conviene comprobar con código lo que dicen las láminas.

### Ejercicio 1
Encuentra pesos `w0_and`, `w1_and` y `w2_and` que sí resuelvan AND (salidas −1, −1, −1, 1). Pista: con $w_1 = w_2 = 1$, ¿qué sesgo hace que solo la suma de (1, 1) sea positiva?

In [ ]:
w0_and, w1_and, w2_and = None, None, None   # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: list(perceptron(X, w0_and, w1_and, w2_and)[1]) == [-1, -1, -1, 1])

## Paso 1.3. El problema XOR (lámina 25)

XOR es verdadero cuando las entradas son **distintas**: (1, 0) y (0, 1). Dibujemos los puntos.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
colores = ["#D9534F", "#1428A0", "#1428A0", "#D9534F"]
ax.scatter(X[:, 0], X[:, 1], c=colores, s=200)
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.grid(alpha=0.3)
ax.set_title("XOR: ninguna recta separa los colores"); plt.show()

**Qué deberías ver:** los azules en una diagonal y los rojos en la otra. **No existe ninguna recta** que deje los azules de un lado y los rojos del otro. Como un perceptrón solo puede dibujar rectas, no puede resolver XOR, lo pruebe con los pesos que lo pruebe.

Esto lo mostraron Minsky y Papert en 1969, y frenó la investigación en redes neuronales por años (láminas 29 y 39).

---
# Parte 2. El perceptrón multicapa resuelve XOR (láminas 28 a 43)

La solución de las láminas 31 y 32: **"doblar el plano"** con una **capa oculta**. En vez de una neurona, usamos dos en paralelo, y una tercera que combina sus salidas.

Los pesos de la lámina 35:

| Neurona | Pesos | Sesgo | Calcula |
|---|---|---|---|
| Oculta 1 ($n_1$) | −2, −2 | 3 | Es 1 salvo que ambas entradas sean 1: **NAND** |
| Oculta 2 ($n_2$) | 2, 2 | −1 | Es 1 si alguna entrada es 1: **OR** |
| Salida ($y$) | 1, 1 | −1 | $n_1 + n_2 - 1$: vale 1 solo si ambas ocultas valen 1: **AND** |

XOR = (NAND) AND (OR): "alguna es 1, pero no las dos".

**Qué vamos a hacer:** calcular la red capa por capa, usando un umbral 0/1 en las neuronas ocultas.

In [ ]:
def escalon01(s):
    return (s >= 0).astype(int)                      # 1 si s >= 0, si no 0

W1 = np.array([[-2, 2],                              # columna 0: pesos de n1; columna 1: pesos de n2
               [-2, 2]])
b1 = np.array([3, -1])
W2 = np.array([1, 1])
b2 = -1

Z1 = X @ W1 + b1                                     # suma de cada neurona oculta (una columna por neurona)
N = escalon01(Z1)                                    # salida de la capa oculta
y = N @ W2 + b2                                      # neurona de salida

pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "suma n1": Z1[:, 0], "suma n2": Z1[:, 1],
              "n1 (NAND)": N[:, 0], "n2 (OR)": N[:, 1], "salida y": y, "XOR esperado": [0, 1, 1, 0]})

**Qué deberías ver:** la salida 0, 1, 1, 0: XOR resuelto.

`X @ W1` multiplica las 4 filas por la matriz de pesos de una vez: es lo que hace una **capa densa**. Así funcionan todas las redes: multiplicaciones de matrices más sesgos, capa tras capa.

## Paso 2.1. ¿Qué significa "doblar el plano"? (láminas 40 y 41)

La capa oculta **transforma** cada punto $(x_1, x_2)$ en un punto nuevo $(n_1, n_2)$. Veamos dónde quedan los 4 puntos en ese espacio nuevo.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(9, 4))
for ax, P, titulo, et in [(ejes[0], X, "Espacio original (x1, x2)", ("x1", "x2")),
                          (ejes[1], N + np.array([[0.03, 0], [0, 0.03], [-0.03, 0], [0, 0]]), "Después de la capa oculta (n1, n2)", ("n1", "n2"))]:
    ax.scatter(P[:, 0], P[:, 1], c=colores, s=200, zorder=3)
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.grid(alpha=0.3); ax.set_title(titulo); ax.set_xlabel(et[0]); ax.set_ylabel(et[1])
xs = np.linspace(-0.5, 1.5, 10)
ejes[1].plot(xs, 1.5 - xs, "--", color="gray")
plt.tight_layout(); plt.show()

**Qué deberías ver:** en el espacio nuevo, los dos azules quedan **en el mismo punto** (1, 1) y los rojos en (1, 0) y (0, 1). Ahora **sí** hay una recta que los separa: la que dibuja la neurona de salida.

**Esta es la idea central de las redes neuronales:** las capas ocultas transforman los datos hasta que el problema se vuelve fácil para la última capa. En las láminas lo llaman "aprendizaje de características": la red aprende sola qué transformación le conviene.

### Ejercicio 2
Cambia solo la **capa de salida** (`W2_xnor` y `b2_xnor`) para que la red calcule **XNOR**, que es lo contrario de XOR (salidas 1, 0, 0, 1). Pista: queremos $1 - (n_1 + n_2 - 1)$.

In [ ]:
W2_xnor, b2_xnor = None, None   # TU CÓDIGO AQUÍ, por ejemplo np.array([..., ...]), ...

In [ ]:
revisar(lambda: list(N @ W2_xnor + b2_xnor) == [1, 0, 0, 1])

## Paso 2.2. Del umbral a la sigmoide (láminas 40 y 43)

Las láminas usan la **sigmoide** en vez del umbral. La diferencia:

- El **umbral** es una "decisión dura": salta de 0 a 1.
- La **sigmoide** $\\sigma(s) = \\frac{1}{1 + e^{-s}}$ es una "decisión suave": pasa de 0 a 1 de forma continua, y su valor se puede leer como **confianza**.

La razón de fondo es que **la sigmoide tiene pendiente** en todos lados, y para entrenar una red necesitamos pendientes (Parte 3). El umbral es plano casi en todas partes: no da información para ajustar los pesos.

In [ ]:
sigmoide = lambda s: 1 / (1 + np.exp(-s))
N_suave = sigmoide(Z1)
pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "n1 sigmoide": N_suave[:, 0].round(3), "n2 sigmoide": N_suave[:, 1].round(3)})

**Qué deberías ver:** los mismos valores de la lámina 35: 0,953; 0,731; 0,731; 0,269 para $n_1$, y 0,269; 0,731; 0,731; 0,953 para $n_2$. Siguen la misma lógica que el umbral, pero con valores intermedios.

---
# Parte 3. Cómo aprende una red (láminas 45 a 59)

En la Parte 2 **nosotros** pusimos los pesos. En la práctica, la red los **aprende** de los datos. El ciclo de entrenamiento (láminas 47 a 51):

1. **Propagación hacia adelante:** con los pesos actuales, calcular la predicción $\\hat{y}$.
2. **Error:** comparar $\\hat{y}$ con el valor real $y$ usando una **función de pérdida**.
3. **Propagación hacia atrás:** calcular cuánto cambia la pérdida si se mueve cada peso: el **gradiente**.
4. **Actualizar:** mover cada peso un poco en la dirección que **baja** la pérdida.
5. Repetir.

## Paso 3.1. Una neurona, a mano

Usamos la red más simple posible: **una neurona sin activación**, $\\hat{y} = w \\cdot x + b$, con 4 datos que siguen la regla $y = 2x + 1$. La red no conoce la regla: parte con $w = 0$ y $b = 0$.

**La pérdida** es el error cuadrático medio (MSE):

$$L = \\frac{1}{n} \\sum (\\hat{y} - y)^2$$

**El gradiente** dice cuánto cambia $L$ al mover cada peso (se obtiene derivando):

$$\\frac{\\partial L}{\\partial w} = \\frac{2}{n} \\sum (\\hat{y} - y) \\cdot x \\qquad \\frac{\\partial L}{\\partial b} = \\frac{2}{n} \\sum (\\hat{y} - y)$$

**La actualización** (descenso de gradiente), con una **tasa de aprendizaje** $\\eta$ que controla el tamaño del paso:

$$w \\leftarrow w - \\eta \\cdot \\frac{\\partial L}{\\partial w} \\qquad b \\leftarrow b - \\eta \\cdot \\frac{\\partial L}{\\partial b}$$

Se **resta** porque el gradiente apunta hacia donde la pérdida **sube**, y queremos bajar.

**Primer paso a mano**, con $w = 0$, $b = 0$ y $\\eta = 0{,}05$:

| x | y real | $\\hat{y}$ | $\\hat{y} - y$ | $(\\hat{y} - y) \\cdot x$ |
|---|---|---|---|---|
| 1 | 3 | 0 | −3 | −3 |
| 2 | 5 | 0 | −5 | −10 |
| 3 | 7 | 0 | −7 | −21 |
| 4 | 9 | 0 | −9 | −36 |
| **Suma** | | | **−24** | **−70** |

- Pérdida: (9 + 25 + 49 + 81) / 4 = **41**
- $\\partial L / \\partial w$ = (2/4) × (−70) = **−35**, entonces $w$ = 0 − 0,05 × (−35) = **1,75**
- $\\partial L / \\partial b$ = (2/4) × (−24) = **−12**, entonces $b$ = 0 − 0,05 × (−12) = **0,6**

Después de un solo paso, $w$ ya se acercó a 2 y $b$ a 1.

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0])
y = np.array([3.0, 5.0, 7.0, 9.0])
w, b, eta = 0.0, 0.0, 0.05

filas = []
for paso in range(6):
    y_hat = w * x + b                                  # 1. hacia adelante
    perdida = np.mean((y_hat - y) ** 2)                # 2. error
    grad_w = 2 * np.mean((y_hat - y) * x)              # 3. gradientes
    grad_b = 2 * np.mean(y_hat - y)
    filas.append({"paso": paso, "w": w, "b": b, "pérdida": perdida, "grad w": grad_w, "grad b": grad_b})
    w -= eta * grad_w                                  # 4. actualizar
    b -= eta * grad_b

pd.DataFrame(filas).round(3)

**Qué deberías ver:** en el paso 0, pérdida 41, gradientes −35 y −12, igual que a mano. En el paso 1, $w = 1{,}75$ y $b = 0{,}6$. La pérdida baja en cada paso.

Si lo dejamos correr más pasos, llega a la regla verdadera:

In [ ]:
w, b = 0.0, 0.0
historial = []
for paso in range(500):
    y_hat = w * x + b
    historial.append(np.mean((y_hat - y) ** 2))
    w -= eta * 2 * np.mean((y_hat - y) * x)
    b -= eta * 2 * np.mean(y_hat - y)

print(f"Después de 500 pasos: w = {w:.3f}, b = {b:.3f}")
plt.plot(historial); plt.yscale("log"); plt.xlabel("Paso"); plt.ylabel("Pérdida (escala log)"); plt.title("La pérdida baja con el descenso de gradiente"); plt.show()

**Qué deberías ver:** $w \\approx 2$ y $b \\approx 1$. La red "descubrió" la regla $y = 2x + 1$ solo con los datos.

**¿Y la retropropagación?** En una red de varias capas, el gradiente de un peso de la primera capa depende de todas las capas que vienen después. La retropropagación calcula esos gradientes **desde la salida hacia atrás**, capa por capa, usando la regla de la cadena del cálculo. Es el mismo paso 3, repetido en cada capa. TensorFlow lo hace automáticamente (Parte 6).

### Ejercicio 3
**Prueba la tasa de aprendizaje.** Repite los 500 pasos con `eta = 0.2` y guarda el `w` final en `w_eta_grande`. ¿Qué pasa? (Pista: mira si es un número razonable.)

In [ ]:
w_eta_grande = None  # TU CÓDIGO AQUÍ
print(w_eta_grande)

In [ ]:
def _correr(eta):
    w, b = 0.0, 0.0
    for _ in range(500):
        y_hat = w * x + b; w -= eta * 2 * np.mean((y_hat - y) * x); b -= eta * 2 * np.mean(y_hat - y)
    return w
revisar(lambda: (np.isnan(w_eta_grande) and np.isnan(_correr(0.2))) or np.isclose(w_eta_grande, _correr(0.2)) or abs(w_eta_grande) > 1e6)

## Paso 3.2. Funciones de activación (láminas 53 a 59)

Sin activación, varias capas seguidas equivalen a una sola capa lineal (multiplicar matrices da otra matriz). La activación agrega la **no linealidad** que permite "doblar el plano".

In [ ]:
s = np.linspace(-5, 5, 200)
activaciones = {
    "Escalón": np.where(s >= 0, 1, 0),
    "Sigmoide": tf.math.sigmoid(s).numpy(),
    "Tanh": tf.math.tanh(s).numpy(),
    "ReLU": tf.nn.relu(s).numpy(),
    "Leaky ReLU": tf.nn.leaky_relu(s, alpha=0.1).numpy(),
}
fig, ejes = plt.subplots(1, 5, figsize=(18, 3.2))
for ax, (nombre, valores) in zip(ejes, activaciones.items()):
    ax.plot(s, valores, color="#1428A0"); ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(nombre); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

| Activación | Salida | Dónde se usa |
|---|---|---|
| Sigmoide | Entre 0 y 1 | Salida de clasificación binaria (una probabilidad) |
| Softmax | Varias salidas que suman 1 | Salida de clasificación multiclase (Clase 15) |
| Tanh | Entre −1 y 1 | Capas ocultas antiguas y redes recurrentes (la LSTM de la Clase 13) |
| ReLU | 0 si es negativo, igual si es positivo | **Capas ocultas: la opción por defecto hoy** |
| Leaky ReLU | Como ReLU, pero con una pequeña pendiente en los negativos | Cuando muchas neuronas ReLU quedan "muertas" en 0 |
| Lineal | Igual a la entrada | Salida de regresión (un número cualquiera) |

---
# Parte 4. Problemas de las redes profundas (láminas 61 a 70)

## Paso 4.1. TensorFlow Playground (láminas 61 y 62)

Abran [playground.tensorflow.org](https://playground.tensorflow.org) y prueben:

1. Elijan el dataset de **XOR** (el de los cuatro cuadrantes). Con 0 capas ocultas, la red no aprende. Agreguen 1 capa oculta con 4 neuronas: aprende.
2. Elijan el dataset de **espiral**. Con activación **lineal**, nunca aprende. Cambien a **ReLU** o **tanh** y agreguen capas.
3. Suban la **tasa de aprendizaje** a 3 o 10: la pérdida se vuelve inestable, como en el Ejercicio 3.

## Paso 4.2. El gradiente que desaparece (láminas 65 y 66)

Con la retropropagación, el gradiente de las primeras capas se obtiene **multiplicando** las derivadas de todas las capas siguientes. La derivada de la sigmoide vale como máximo **0,25**. Si se multiplican muchos valores menores que 1, el resultado se acerca a cero.

In [ ]:
s = np.linspace(-6, 6, 200)
sig = 1 / (1 + np.exp(-s))
derivada_sig = sig * (1 - sig)
print("Derivada máxima de la sigmoide:", derivada_sig.max().round(3))

capas = np.arange(1, 21)
plt.figure(figsize=(7, 4))
plt.plot(capas, 0.25 ** capas, marker="o", label="Sigmoide: 0,25 por capa (en el mejor caso)")
plt.plot(capas, 1.0 ** capas, marker="o", label="ReLU: derivada 1 en la parte positiva")
plt.yscale("log"); plt.xlabel("Número de capas"); plt.ylabel("Tamaño del gradiente (log)")
plt.title("El gradiente que llega a la primera capa"); plt.legend(); plt.show()
print("Con 10 capas sigmoide, el gradiente se multiplica por:", 0.25 ** 10)

**Qué deberías ver:** con sigmoide, el gradiente se reduce a la cuarta parte en cada capa: con 10 capas queda cerca de una millonésima. Las primeras capas casi no aprenden. **Por eso las capas ocultas usan ReLU**, cuya derivada es 1 en la parte positiva.

### Ejercicio 4
¿Desde cuántas capas sigmoide el factor $0{,}25^n$ queda por debajo de **una millonésima** (1e-6)? Guárdalo en `capas_millonesima`.

In [ ]:
capas_millonesima = None  # TU CÓDIGO AQUÍ
print(capas_millonesima)

In [ ]:
revisar(lambda: capas_millonesima == 10)

## Paso 4.3. Sobreajuste y dropout (láminas 67 a 70)

Las redes profundas tienen muchísimos parámetros y tienden a **memorizar** los datos de entrenamiento (lo vimos con la LSTM de reseñas en la Clase 13).

> **Dropout (abandono):** durante el entrenamiento, **apaga al azar** una fracción de las neuronas en cada paso. Así ninguna neurona puede depender de otra específica, y la red aprende patrones más robustos. Al **predecir**, se usan todas (lámina 70).

**Qué vamos a hacer:** pasar un vector de unos por una capa `Dropout(0.5)`, en modo entrenamiento y en modo predicción.

In [ ]:
tf.random.set_seed(0)
capa_dropout = tf.keras.layers.Dropout(0.5)
entrada = tf.ones((1, 10))

print("Entrenando (training=True): ", capa_dropout(entrada, training=True).numpy())
print("Prediciendo (training=False):", capa_dropout(entrada, training=False).numpy())

**Qué deberías ver:** entrenando, cerca de la mitad de los valores quedan en 0 y los demás valen **2**. Prediciendo, todos valen 1.

¿Por qué 2? Si se apaga la mitad, los que quedan se multiplican por 1 / (1 − 0,5) = 2, para que la suma total se mantenga parecida entre entrenamiento y predicción.

---
# Parte 5. TensorFlow esencial (láminas 72 a 109)

> **Tensor:** un arreglo de números de cualquier número de dimensiones (lámina 78). Un número es un tensor de 0 dimensiones; un vector, de 1; una matriz, de 2; una imagen a color, de 3 (alto, ancho, canales); un lote de imágenes, de 4.

Es la misma idea de los arreglos de NumPy, con dos ventajas: TensorFlow puede usar **GPU** y puede calcular **gradientes automáticamente** (láminas 75 a 77).

## Paso 5.1. Crear tensores (láminas 87 a 89)

In [ ]:
escalar = tf.constant(3.0)
vector = tf.constant([1, 2, 3])
matriz = tf.constant([[1.0, 2.0], [3.0, 4.0]])
desde_numpy = tf.convert_to_tensor(np.arange(6))

for nombre, t in [("escalar", escalar), ("vector", vector), ("matriz", matriz), ("desde NumPy", desde_numpy)]:
    print(f"{nombre:12s} forma={t.shape}  tipo={t.dtype.name}  dimensiones={t.ndim}")

peso = tf.Variable([0.5, -0.5])          # Variable: un tensor que puede cambiar (los pesos de una red)
peso.assign([1.0, 1.0])
print("\nVariable después de assign:", peso.numpy())

**Qué deberías ver:** formas `()`, `(3,)`, `(2, 2)` y `(6,)`, y tipos `float32` o `int32`/`int64`.

- `tf.constant`: un tensor fijo.
- `tf.Variable`: un tensor que **cambia** durante el entrenamiento. Los pesos de una red son variables.

## Paso 5.2. Forma y tipo (láminas 90 y 91)

In [ ]:
t = tf.range(12)
print("Original:          ", t.numpy(), t.shape)
print("reshape a (3, 4):\n", tf.reshape(t, (3, 4)).numpy())
print("transpose de (3, 4) queda", tf.transpose(tf.reshape(t, (3, 4))).shape)
print("cast a float32:    ", tf.cast(t, tf.float32).dtype.name)
print("expand_dims:       ", tf.expand_dims(t, axis=0).shape, "(agrega una dimensión: útil para hacer un 'lote' de 1)")

## Paso 5.3. Operaciones (láminas 92 a 97)

In [ ]:
A = tf.constant([[1.0, 2.0], [3.0, 4.0]])
B = tf.constant([[1.0, 0.0], [0.0, 2.0]])

print("A * B (elemento a elemento):\n", (A * B).numpy())
print("A @ B (producto de matrices):\n", tf.matmul(A, B).numpy())
print("Suma total:", tf.reduce_sum(A).numpy(), "| Promedio por columna:", tf.reduce_mean(A, axis=0).numpy())
print("A NumPy y de vuelta:", type(A.numpy()).__name__, "->", type(tf.constant(A.numpy())).__name__)

partes = tf.split(tf.range(6), num_or_size_splits=3)
print("\nsplit en 3:", [p.numpy() for p in partes])
print("stack (nueva dimensión):", tf.stack(partes).shape, "| concat (misma dimensión):", tf.concat(partes, axis=0).shape)

**Qué deberías ver:** la diferencia entre `*` (multiplica posición por posición) y `@` o `tf.matmul` (producto de matrices, el que usa una capa densa). `stack` apila creando una dimensión nueva; `concat` une a lo largo de una existente.

### Ejercicio 5
Con `tf.range(12)`, crea una matriz de **4 filas y 3 columnas**, y guarda en `suma_columnas` la **suma de cada columna** como arreglo de NumPy.

In [ ]:
suma_columnas = None  # TU CÓDIGO AQUÍ
print(suma_columnas)

In [ ]:
revisar(lambda: np.array_equal(suma_columnas, np.array([18, 22, 26])))

## Paso 5.4. `tf.data`: alimentar a la red por lotes (láminas 100 a 109)

Las redes no se entrenan con todos los datos a la vez, sino por **lotes** (*batches*). `tf.data.Dataset` organiza ese flujo:

| Método | Qué hace |
|---|---|
| `from_tensor_slices((X, y))` | Crea un dataset con un elemento por fila |
| `shuffle(n)` | Desordena (con un "balde" de n elementos) |
| `batch(k)` | Agrupa en lotes de k elementos |
| `repeat(e)` | Repite el dataset e veces (épocas) |
| `map(f)` | Aplica una función a cada elemento |

In [ ]:
X_d = tf.range(10)
y_d = X_d * 2
ds = tf.data.Dataset.from_tensor_slices((X_d, y_d))
print("Primeros 3 elementos:", [(int(a), int(b)) for a, b in ds.take(3)])

print("\nEn lotes de 4:")
for lote_x, lote_y in ds.batch(4):
    print("  x:", lote_x.numpy(), "| y:", lote_y.numpy())

print("\nDesordenado y en lotes de 4 (semilla fija):")
for lote_x, _ in ds.shuffle(10, seed=1).batch(4):
    print("  x:", lote_x.numpy())

**Qué deberías ver:** 3 lotes (de 4, 4 y 2 elementos) y luego los mismos números desordenados. En cada época, `shuffle` cambia el orden para que la red no aprenda nada del orden de los datos.

### Ejercicio 6
Con `ds` (10 elementos), ¿cuántos lotes produce `ds.batch(3, drop_remainder=True)`? Guárdalo en `n_lotes`. (`drop_remainder=True` descarta el último lote si queda incompleto.)

In [ ]:
n_lotes = None  # TU CÓDIGO AQUÍ
print(n_lotes)

In [ ]:
revisar(lambda: n_lotes == 3)

---
# Parte 6. Primer modelo con TensorFlow (láminas 130 a 147)

Repetimos la neurona de la Parte 3, pero ahora **TensorFlow calcula los gradientes** con `tf.GradientTape`.

> **`GradientTape` (cinta de gradientes):** "graba" las operaciones que se hacen dentro de un bloque `with`, y después calcula automáticamente el gradiente de cualquier resultado respecto de las variables. Es la retropropagación hecha por TensorFlow.

## Paso 6.1. Comprobar que el gradiente es el mismo

In [ ]:
x_t = tf.constant([1.0, 2.0, 3.0, 4.0])
y_t = tf.constant([3.0, 5.0, 7.0, 9.0])
w_t = tf.Variable(0.0)
b_t = tf.Variable(0.0)

with tf.GradientTape() as cinta:
    y_hat = w_t * x_t + b_t
    perdida = tf.reduce_mean((y_hat - y_t) ** 2)

grad_w, grad_b = cinta.gradient(perdida, [w_t, b_t])
print(f"Pérdida: {perdida.numpy():.1f} | gradiente de w: {grad_w.numpy():.1f} | gradiente de b: {grad_b.numpy():.1f}")

**Qué deberías ver:** pérdida 41, gradientes −35 y −12. **Exactamente lo que calculamos a mano en la Parte 3**, pero sin escribir ninguna derivada.

## Paso 6.2. El ciclo de entrenamiento completo

Ahora con datos más realistas: 200 puntos con ruido alrededor de $y = 2x + 1$, entregados por lotes con `tf.data`, y un **optimizador** que hace la actualización por nosotros.

In [ ]:
rng = np.random.default_rng(42)
x_np = rng.uniform(0, 10, 200).astype("float32")
y_np = (2 * x_np + 1 + rng.normal(0, 1.5, 200)).astype("float32")

datos = tf.data.Dataset.from_tensor_slices((x_np, y_np)).shuffle(200, seed=42).batch(20)

w_t = tf.Variable(0.0)
b_t = tf.Variable(0.0)
optimizador = tf.keras.optimizers.SGD(learning_rate=0.01)

historial = []
for epoca in range(50):
    for lote_x, lote_y in datos:
        with tf.GradientTape() as cinta:                              # 1. hacia adelante, grabando
            perdida = tf.reduce_mean((w_t * lote_x + b_t - lote_y) ** 2)   # 2. error
        gradientes = cinta.gradient(perdida, [w_t, b_t])              # 3. retropropagación automática
        optimizador.apply_gradients(zip(gradientes, [w_t, b_t]))      # 4. actualizar
    historial.append(perdida.numpy())

w_mc, b_mc = np.polyfit(x_np, y_np, 1)                               # la mejor recta posible (mínimos cuadrados)
print(f"Aprendido con SGD:      w = {w_t.numpy():.3f}, b = {b_t.numpy():.3f}")
print(f"Mínimos cuadrados:      w = {w_mc:.3f}, b = {b_mc:.3f}")
print("Regla con que se generaron los datos: w = 2, b = 1")
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(historial); ejes[0].set_title("Pérdida por época"); ejes[0].set_xlabel("Época")
ejes[1].scatter(x_np, y_np, s=10, alpha=0.5); xs = np.linspace(0, 10, 10)
ejes[1].plot(xs, w_t.numpy() * xs + b_t.numpy(), color="red"); ejes[1].set_title("Datos y recta aprendida")
plt.tight_layout(); plt.show()

**Qué deberías ver:** la pérdida bajando y una recta que pasa por el medio de los puntos. El $w$ aprendido (cerca de 2,13) y el $b$ (cerca de 0,52) no son exactamente 2 y 1: por el ruido, **la mejor recta posible para estos 200 puntos** (mínimos cuadrados, la de la regresión lineal del Capítulo 5) es $w = 2{,}10$ y $b = 0{,}54$. El descenso de gradiente llegó muy cerca de ella.

**Las palabras nuevas:**

- **Época:** una pasada completa por todos los datos. Aquí, 50 épocas.
- **Lote:** el grupo de datos con que se calcula cada actualización. Aquí, 20 puntos: 10 actualizaciones por época.
- **SGD:** descenso de gradiente estocástico, el mismo descenso de gradiente pero calculado con lotes en vez de todos los datos.

En la **Clase 15**, Keras hará todo este ciclo con una sola línea: `modelo.fit(...)`. Pero por dentro hace exactamente esto.

### Ejercicio 7
Guarda en `diferencia_w` la diferencia, en valor absoluto, entre el `w` aprendido con SGD (`w_t`) y el de mínimos cuadrados (`w_mc`). Debería ser menor que 0,1.

In [ ]:
diferencia_w = None  # TU CÓDIGO AQUÍ
print(diferencia_w)

In [ ]:
revisar(lambda: np.isclose(diferencia_w, abs(w_t.numpy() - w_mc)) and diferencia_w < 0.1)

---
# Cierre

| Parte | Lo esencial |
|---|---|
| Perceptrón | Suma ponderada más sesgo, y un umbral. Dibuja una recta: resuelve OR y AND, no XOR. |
| Perceptrón multicapa | La capa oculta transforma los datos hasta que una recta basta. |
| Aprender | Hacia adelante, error, gradiente, actualizar. La tasa de aprendizaje controla el paso. |
| Activaciones | ReLU en capas ocultas; sigmoide o softmax en la salida de clasificación; lineal en regresión. |
| Problemas | Gradiente que desaparece (usar ReLU) y sobreajuste (usar dropout). |
| TensorFlow | Tensores, variables, `tf.data` por lotes y `GradientTape` para los gradientes. |

Guarden el notebook en `ejercicios_estudiantes` como `clase14_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1
Con $w_1 = w_2 = 1$, las sumas sin sesgo son 0, 1, 1 y 2. Un sesgo de −1,5 deja negativas las tres primeras y positiva solo la de (1, 1).

In [ ]:
w0_and, w1_and, w2_and = -1.5, 1, 1
print(perceptron(X, w0_and, w1_and, w2_and))

### Ejercicio 2
$1 - (n_1 + n_2 - 1) = -n_1 - n_2 + 2$.

In [ ]:
W2_xnor, b2_xnor = np.array([-1, -1]), 2
print(N @ W2_xnor + b2_xnor)

### Ejercicio 3
Con `eta = 0.2` el paso es tan grande que cada actualización se pasa del mínimo y cae más lejos del otro lado: los pesos crecen sin control hasta valores enormes o `nan`. Es una tasa de aprendizaje demasiado alta.

In [ ]:
w, b = 0.0, 0.0
for _ in range(500):
    y_hat = w * x + b
    w -= 0.2 * 2 * np.mean((y_hat - y) * x)
    b -= 0.2 * 2 * np.mean(y_hat - y)
w_eta_grande = w
print(w_eta_grande)

### Ejercicio 4
$0{,}25^9 \\approx 3{,}8 \\times 10^{-6}$ y $0{,}25^{10} \\approx 9{,}5 \\times 10^{-7}$.

In [ ]:
capas_millonesima = next(n for n in range(1, 50) if 0.25 ** n < 1e-6)
print(capas_millonesima)

### Ejercicio 5

In [ ]:
suma_columnas = tf.reduce_sum(tf.reshape(tf.range(12), (4, 3)), axis=0).numpy()
print(suma_columnas)

### Ejercicio 6
10 elementos en lotes de 3: 3 lotes completos y 1 elemento que se descarta.

In [ ]:
n_lotes = len(list(ds.batch(3, drop_remainder=True)))
print(n_lotes)

### Ejercicio 7

In [ ]:
diferencia_w = abs(float(w_t.numpy()) - w_mc)
print(round(diferencia_w, 3))